In [0]:
# ============================================================
# nb_silver_to_gold — FIXED INCREMENTAL
# Tracks last processed timestamp in a control file
# Only processes Silver rows newer than last Gold run
# ============================================================

from pyspark.sql.functions import (
    col, sum, count, avg, countDistinct,
    current_timestamp, lit, max as spark_max
)
from delta.tables import DeltaTable
from datetime import datetime

# ── Paths ───────────────────────────────────────────────────
SILVER_PATH  = "abfss://silver@ecommerceprojectstorage.dfs.core.windows.net/ecommerce_sales/"
GOLD_PATH    = "abfss://gold@ecommerceprojectstorage.dfs.core.windows.net/"
CONTROL_PATH = "abfss://gold@ecommerceprojectstorage.dfs.core.windows.net/_control/last_gold_run/"

# ── Step 1: Read Silver ─────────────────────────────────────
df_silver = spark.read.format("delta").load(SILVER_PATH)

# ── Step 2: Get last Gold run timestamp from control file ───
try:
    df_control = spark.read.format("delta").load(CONTROL_PATH)
    last_run_ts = df_control.collect()[0]["last_run_ts"]
    print(f"Last Gold run timestamp: {last_run_ts}")
    
    # Filter Silver rows loaded AFTER last Gold run
    df_new = df_silver.filter(col("_silver_load_ts") > lit(last_run_ts))

except Exception:
    # First run — no control file yet, process everything
    print("No control file found. Processing all Silver rows (first run).")
    df_new = df_silver

new_count = df_new.count()
print(f"New Silver rows to process: {new_count}")

if new_count == 0:
    print("No new Silver data since last Gold run. Exiting.")
    dbutils.notebook.exit("NO_NEW_DATA")

# ── Step 3: Helper — merge into Gold Delta table ────────────
def merge_to_gold(new_df, path, merge_keys):
    row_count = new_df.count()
    if row_count == 0:
        print(f"Skipping — no data for {path}")
        return

    if DeltaTable.isDeltaTable(spark, path):
        condition = " AND ".join(
            [f"tgt.{k} = src.{k}" for k in merge_keys]
        )
        DeltaTable.forPath(spark, path).alias("tgt") \
            .merge(new_df.alias("src"), condition) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        print(f"MERGE done ({row_count} rows) → {path}")
    else:
        new_df.write.format("delta") \
            .mode("overwrite").save(path)
        print(f"Initial write done ({row_count} rows) → {path}")


# ── Step 4: Gold Table 1 — Monthly Sales ────────────────────
gold_monthly = df_new.groupBy(
    "order_year", "order_month", "order_quarter",
    "category", "region"
).agg(
    sum("sales").alias("total_sales"),
    sum("net_sales").alias("total_net_sales"),
    sum("profit").alias("total_profit"),
    count("order_id").alias("total_orders"),
    avg("discount").alias("avg_discount_pct"),
    avg("profit_margin_pct").alias("avg_margin_pct"),
    countDistinct("customer_name").alias("unique_customers")
).withColumn("_gold_load_ts", current_timestamp())

merge_to_gold(
    new_df     = gold_monthly,
    path       = GOLD_PATH + "monthly_sales_by_category/",
    merge_keys = ["order_year", "order_month", "category", "region"]
)


# ── Step 5: Gold Table 2 — Payment Analysis ─────────────────
gold_payment = df_new.groupBy(
    "payment_mode", "payment_status", "region"
).agg(
    count("order_id").alias("total_orders"),
    sum("sales").alias("total_value"),
    avg("sales").alias("avg_order_value")
).withColumn("_gold_load_ts", current_timestamp())

merge_to_gold(
    new_df     = gold_payment,
    path       = GOLD_PATH + "payment_analysis/",
    merge_keys = ["payment_mode", "payment_status", "region"]
)


# ── Step 6: Gold Table 3 — Top Products ─────────────────────
gold_products = df_new.groupBy(
    "category", "sub_category", "product_name"
).agg(
    sum("quantity").alias("total_qty_sold"),
    sum("sales").alias("total_revenue"),
    sum("profit").alias("total_profit"),
    avg("profit_margin_pct").alias("avg_margin_pct")
).withColumn("_gold_load_ts", current_timestamp())

merge_to_gold(
    new_df     = gold_products,
    path       = GOLD_PATH + "top_products/",
    merge_keys = ["category", "sub_category", "product_name"]
)


# ── Step 7: Update control file with current timestamp ──────
# Next run will only pick up Silver rows after this moment
current_ts = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

control_df = spark.createDataFrame(
    [(current_ts,)],
    ["last_run_ts"]
)

control_df.write.format("delta") \
    .mode("overwrite") \
    .save(CONTROL_PATH)

print(f"\nControl file updated: last_run_ts = {current_ts}")
print("nb_silver_to_gold DONE ✅")
print(f"Processed {new_count} new rows into Gold tables")